In [ ]:
# Load and use the simple MLB prediction model
import os
import joblib
import pandas as pd
import numpy as np
from datetime import datetime

def load_mlb_simple_model():
    """
    Load the pre-trained simple MLB model, scaler and feature list
    """
    try:
        # Find the latest model files (you may need to adjust the paths)
        import glob
        model_files = glob.glob('models/mlb_simple_model_*.joblib')
        
        if not model_files:
            print("No model files found. Please check the path.")
            return None, None, None
        
        # Get the latest model
        latest_model = max(model_files, key=os.path.getctime)
        
        # Load corresponding scaler and features
        timestamp = '_'.join(latest_model.split('_')[-2:]).replace('.joblib', '')
        scaler_path = f'models/mlb_simple_scaler_{timestamp}.joblib'
        features_path = f'models/mlb_simple_features_{timestamp}.joblib'
        
        model = joblib.load(latest_model)
        scaler = joblib.load(scaler_path)
        features = joblib.load(features_path)
        
        print(f"Successfully loaded model: {latest_model}")
        print(f"Features used: {features}")
        
        return model, scaler, features
        
    except Exception as e:
        print(f"Error loading model: {e}")
        return None, None, None

def predict_mlb_game(home_ba, away_ba, home_slg, away_slg, home_obp, away_obp, model, scaler, features):
    """
    Predict MLB game outcome using the simple model
    """
    # Create input dataframe
    input_data = pd.DataFrame({
        'home_batting_avg': [home_ba],
        'visiting_batting_avg': [away_ba],
        'home_slugging': [home_slg],
        'visiting_slugging': [away_slg],
        'home_obp': [home_obp],
        'visiting_obp': [away_obp]
    })[features]  # Ensure correct feature order
    
    # Scale the input
    input_scaled = scaler.transform(input_data)
    
    # Make prediction
    prediction_proba = model.predict_proba(input_scaled)[0]
    prediction_class = model.predict(input_scaled)[0]
    
    return prediction_proba, prediction_class

def create_dummy_predictions():
    """
    Create predictions for dummy data scenarios
    """
    # Load the model
    model, scaler, features = load_mlb_simple_model()
    
    if model is None:
        print("Could not load model. Please train the model first.")
        return
    
    print("\n" + "="*60)
    print("MLB GAME PREDICTIONS - DUMMY DATA SCENARIOS")
    print("="*60)
    
    # Define dummy scenarios
    scenarios = [
        {
            'name': 'Strong Home vs Weak Away',
            'home_ba': 0.280, 'away_ba': 0.230,
            'home_slg': 0.480, 'away_slg': 0.380,
            'home_obp': 0.350, 'away_obp': 0.300
        },
        {
            'name': 'Even Matchup',
            'home_ba': 0.260, 'away_ba': 0.260,
            'home_slg': 0.430, 'away_slg': 0.430,
            'home_obp': 0.330, 'away_obp': 0.330
        },
        {
            'name': 'Weak Home vs Strong Away',
            'home_ba': 0.240, 'away_ba': 0.270,
            'home_slg': 0.400, 'away_slg': 0.450,
            'home_obp': 0.310, 'away_obp': 0.340
        },
        {
            'name': 'Power Hitting Home Team',
            'home_ba': 0.250, 'away_ba': 0.260,
            'home_slg': 0.500, 'away_slg': 0.420,
            'home_obp': 0.320, 'away_obp': 0.330
        },
        {
            'name': 'High OBP Away Team',
            'home_ba': 0.260, 'away_ba': 0.255,
            'home_slg': 0.440, 'away_slg': 0.430,
            'home_obp': 0.325, 'away_obp': 0.360
        }
    ]
    
    results = []
    
    for scenario in scenarios:
        proba, pred_class = predict_mlb_game(
            scenario['home_ba'], scenario['away_ba'],
            scenario['home_slg'], scenario['away_slg'],
            scenario['home_obp'], scenario['away_obp'],
            model, scaler, features
        )
        
        home_win_prob = proba[1] * 100  # Convert to percentage
        away_win_prob = proba[0] * 100
        
        result = {
            'Scenario': scenario['name'],
            'Home Win %': f"{home_win_prob:.1f}%",
            'Away Win %': f"{away_win_prob:.1f}%",
            'Predicted Winner': 'Home' if pred_class == 1 else 'Away',
            'Home BA': scenario['home_ba'],
            'Away BA': scenario['away_ba'],
            'Home SLG': scenario['home_slg'],
            'Away SLG': scenario['away_slg'],
            'Home OBP': scenario['home_obp'],
            'Away OBP': scenario['away_obp']
        }
        
        results.append(result)
        
        print(f"\n{scenario['name']}:")
        print(f"  Home: BA={scenario['home_ba']}, SLG={scenario['home_slg']}, OBP={scenario['home_obp']}")
        print(f"  Away: BA={scenario['away_ba']}, SLG={scenario['away_slg']}, OBP={scenario['away_obp']}")
        print(f"  Prediction: Home {home_win_prob:.1f}% - Away {away_win_prob:.1f}%")
        print(f"  Expected Winner: {'Home' if pred_class == 1 else 'Away'}")
    
    # Create results dataframe
    results_df = pd.DataFrame(results)
    print("\n" + "="*60)
    print("SUMMARY OF ALL PREDICTIONS")
    print("="*60)
    print(results_df.to_string(index=False))
    
    return results_df

# Also create a function for single prediction with custom inputs
def predict_custom_game():
    """
    Make prediction with custom user input
    """
    model, scaler, features = load_mlb_simple_model()
    
    if model is None:
        return
    
    print("\nEnter custom team statistics:")
    
    try:
        home_ba = float(input("Home Team Batting Average (e.g., 0.265): "))
        away_ba = float(input("Away Team Batting Average (e.g., 0.255): "))
        home_slg = float(input("Home Team Slugging (e.g., 0.440): "))
        away_slg = float(input("Away Team Slugging (e.g., 0.420): "))
        home_obp = float(input("Home Team OBP (e.g., 0.330): "))
        away_obp = float(input("Away Team OBP (e.g., 0.320): "))
        
        proba, pred_class = predict_mlb_game(
            home_ba, away_ba, home_slg, away_slg, home_obp, away_obp,
            model, scaler, features
        )
        
        home_win_prob = proba[1] * 100
        away_win_prob = proba[0] * 100
        
        print("\n" + "="*40)
        print("CUSTOM PREDICTION RESULTS")
        print("="*40)
        print(f"Home Team Win Probability: {home_win_prob:.1f}%")
        print(f"Away Team Win Probability: {away_win_prob:.1f}%")
        print(f"Predicted Winner: {'Home Team' if pred_class == 1 else 'Away Team'}")
        
    except ValueError:
        print("Please enter valid numbers (e.g., 0.265)")

# Execute the predictions
#print("Loading MLB prediction model and creating dummy predictions...")
#results = create_dummy_predictions()

# predict_custom_game()

#print("\nPrediction completed! Use predict_custom_game() for custom inputs.")

In [ ]:
# Get recent team stats

import requests
import pandas as pd

MLB_TEAM_IDS = {
    108: {'abbreviation': 'LAA', 'name': 'Los Angeles Angels', 'league': 'AL', 'division': 'West'},
    109: {'abbreviation': 'ARI', 'name': 'Arizona Diamondbacks', 'league': 'NL', 'division': 'West'},
    110: {'abbreviation': 'BAL', 'name': 'Baltimore Orioles', 'league': 'AL', 'division': 'East'},
    111: {'abbreviation': 'BOS', 'name': 'Boston Red Sox', 'league': 'AL', 'division': 'East'},
    112: {'abbreviation': 'CHC', 'name': 'Chicago Cubs', 'league': 'NL', 'division': 'Central'},
    113: {'abbreviation': 'CIN', 'name': 'Cincinnati Reds', 'league': 'NL', 'division': 'Central'},
    114: {'abbreviation': 'CLE', 'name': 'Cleveland Guardians', 'league': 'AL', 'division': 'Central'},
    115: {'abbreviation': 'COL', 'name': 'Colorado Rockies', 'league': 'NL', 'division': 'West'},
    116: {'abbreviation': 'DET', 'name': 'Detroit Tigers', 'league': 'AL', 'division': 'Central'},
    117: {'abbreviation': 'HOU', 'name': 'Houston Astros', 'league': 'AL', 'division': 'West'},
    118: {'abbreviation': 'KCR', 'name': 'Kansas City Royals', 'league': 'AL', 'division': 'Central'},
    119: {'abbreviation': 'LAD', 'name': 'Los Angeles Dodgers', 'league': 'NL', 'division': 'West'},
    120: {'abbreviation': 'WSN', 'name': 'Washington Nationals', 'league': 'NL', 'division': 'East'},
    121: {'abbreviation': 'NYM', 'name': 'New York Mets', 'league': 'NL', 'division': 'East'},
    133: {'abbreviation': 'OAK', 'name': 'Oakland Athletics', 'league': 'AL', 'division': 'West'},
    134: {'abbreviation': 'PIT', 'name': 'Pittsburgh Pirates', 'league': 'NL', 'division': 'Central'},
    135: {'abbreviation': 'SDP', 'name': 'San Diego Padres', 'league': 'NL', 'division': 'West'},
    136: {'abbreviation': 'SEA', 'name': 'Seattle Mariners', 'league': 'AL', 'division': 'West'},
    137: {'abbreviation': 'SFG', 'name': 'San Francisco Giants', 'league': 'NL', 'division': 'West'},
    138: {'abbreviation': 'STL', 'name': 'St. Louis Cardinals', 'league': 'NL', 'division': 'Central'},
    139: {'abbreviation': 'TBR', 'name': 'Tampa Bay Rays', 'league': 'AL', 'division': 'East'},
    140: {'abbreviation': 'TEX', 'name': 'Texas Rangers', 'league': 'AL', 'division': 'West'},
    141: {'abbreviation': 'TOR', 'name': 'Toronto Blue Jays', 'league': 'AL', 'division': 'East'},
    142: {'abbreviation': 'MIN', 'name': 'Minnesota Twins', 'league': 'AL', 'division': 'Central'},
    143: {'abbreviation': 'PHI', 'name': 'Philadelphia Phillies', 'league': 'NL', 'division': 'East'},
    144: {'abbreviation': 'ATL', 'name': 'Atlanta Braves', 'league': 'NL', 'division': 'East'},
    145: {'abbreviation': 'CHW', 'name': 'Chicago White Sox', 'league': 'AL', 'division': 'Central'},
    146: {'abbreviation': 'MIA', 'name': 'Miami Marlins', 'league': 'NL', 'division': 'East'},
    147: {'abbreviation': 'NYY', 'name': 'New York Yankees', 'league': 'AL', 'division': 'East'},
    158: {'abbreviation': 'MIL', 'name': 'Milwaukee Brewers', 'league': 'NL', 'division': 'Central'}
}

MLB_TEAM_IDS_BY_ABBR = {v['abbreviation']: {'id': k, **v} for k, v in MLB_TEAM_IDS.items()}
MLB_TEAM_IDS_BY_NAME = {v['name']: {'id': k, **v} for k, v in MLB_TEAM_IDS.items()}


def get_team_recent_stats(team_id, games=5):
    """
    Get recent batting stats for a team using MLB Stats API
    """
    url = f"https://statsapi.mlb.com/api/v1/teams/{team_id}/stats"
    params = {
        'season': 2025,
        'stats': 'gameLog',
        'group': 'hitting',
    }
    
    response = requests.get(url, params=params)
    data = response.json()
    
    # Process the data to calculate averages
    stats = {
        'avg': [], 'slg': [], 'obp': []
    }
    
    for game in data['stats'][0]['splits'][-games:]:
        game_stats = game['stat']
        stats['avg'].append(float(game_stats.get('avg', 0)))
        stats['obp'].append(float(game_stats.get('obp', 0)))
        stats['slg'].append(float(game_stats.get('slg', 0)))
    
    # Calculate rolling averages
    recent_avg = sum(stats['avg']) / len(stats['avg'])
    recent_obp = sum(stats['obp']) / len(stats['obp'])
    recent_slg = sum(stats['slg']) / len(stats['slg'])
    
    return {
        'avg': round(recent_avg, 3),
        'obp': round(recent_obp, 3),
        'slg': round(recent_slg, 3),
        'games': games
    }

team_abb = "SEA"
team_id = MLB_TEAM_IDS_BY_ABBR[team_abb]['id']

get_team_recent_stats(team_id)

In [ ]:
# Get upcoming MLB games
import requests
from datetime import datetime, timedelta

def get_upcoming_mlb_games(span=3):
    """
    Simple function to get upcoming MLB games with team IDs
    """
    url = "https://statsapi.mlb.com/api/v1/schedule/games/"
    params = {
        'sportId': 1,  # MLB sport ID
        'startDate': datetime.now().strftime('%Y-%m-%d'),
        'endDate': (datetime.now() + timedelta(days=span)).strftime('%Y-%m-%d')
    }
    
    try:
        response = requests.get(url, params=params, timeout=10)
        data = response.json()
        
        upcoming_games = []
        
        for date in data['dates']:
            for game in date['games']:
                game_info = {
                    'game_id': game['gamePk'],
                    'date': date['date'],
                    'away_team_id': game['teams']['away']['team']['id'],
                    'away_team_name': game['teams']['away']['team']['name'],
                    'home_team_id': game['teams']['home']['team']['id'],
                    'home_team_name': game['teams']['home']['team']['name']
                }
                upcoming_games.append(game_info)
        
        return pd.DataFrame(upcoming_games)
        
    except Exception as e:
        print(f"Error: {e}")
        return []

# Használat
games = get_upcoming_mlb_games(0)
print(f"{len(games)} games found:")
for _, game in games.iterrows():
    print(f"{game['away_team_name']} ({game['away_team_id']}) @ {game['home_team_name']} ({game['home_team_id']}) - {game['date']}")

In [ ]:
# Predict

model, scaler, features = load_mlb_simple_model()

for i, game in games.iterrows():
    last5 = {}
    for team in ['home', 'away']:
        last5[f"{team}_stats"] = get_team_recent_stats(game[f'{team}_team_id'])
    
    avg_home = last5['home_stats']['avg']
    avg_away = last5['away_stats']['avg']
    slg_home = last5['home_stats']['slg']
    slg_away = last5['away_stats']['slg']
    obp_home = last5['home_stats']['obp']
    obp_away = last5['away_stats']['obp']
        
    proba, pred_class = predict_mlb_game(avg_home, avg_away, slg_home, slg_away, obp_home, obp_away, model, scaler, features)
    games.loc[i, 'away_prob'] = proba[0]
    games.loc[i, 'home_prob'] = proba[1]
    games.loc[i, 'away_implied_prob'] = 1 / proba[0]
    games.loc[i, 'home_implied_prob'] = 1 / proba[1]

# Print predictions
print(f"{len(games)} games")
print("="*40)
for i, game in games.iterrows():
    print(f"{game['home_team_name']} vs {game['away_team_name']} - {game['date']}")
    print(f"Home: {game['home_prob']:.0%} ({game['home_implied_prob']:.2f})")
    print(f"Away: {game['away_prob']:.0%} ({game['away_implied_prob']:.2f})")
    print("="*40)

In [ ]:
# Get tippmix odds

import requests
import json
import pandas as pd
from datetime import datetime, timedelta, time
import numpy as np

url = 'https://api.tippmix.hu/event'
response = requests.get(url)

if response.status_code != 200:
    print(f"Hiba történt: {response.status_code}")

try:
    response.raise_for_status()

    data = response.json()
    if data is None:
        print("No data received")

    matches = data['data']

    today_date = datetime.today().date()
    span = timedelta(days=1)


    matches_f = []
    for match in matches:
        match_date_iso = datetime.fromisoformat(match['eventDate'])
        match_date = match_date_iso.date()
        filter_ = (
            (match['sportId'] == 3) and
            (match['competitionName'] == 'MLB') and 
            (match_date <= today_date+span)
            )
        if filter_:
            matches_f.append(match)

    # Find odds of filtered matches
    odds = []
    for match_f in matches_f:
        match_odds = {}
        match_date_iso = datetime.fromisoformat(match_f['eventDate'])
        match_odds['Date'] = datetime.combine(match_date_iso.date(), match_date_iso.time())
        match_odds['Home'] = match_f['eventParticipants'][0]['participantName']
        match_odds['Away'] = match_f['eventParticipants'][1]['participantName']
        
        found_1X2 = False
        for market in match_f['markets']:
            # Get 1X2 odds
            if market['marketName'] == 'A mérkőzés győztese':
                match_odds['H_odds'] = market['outcomes'][0]['fixedOdds']
                match_odds['A_odds'] = market['outcomes'][1]['fixedOdds']
                found_1X2 = True
            else:
                pass
            
        if found_1X2:
            odds.append(match_odds)

    df_odds = pd.DataFrame(odds)

    print(df_odds.head())

except requests.exceptions.HTTPError as e:
    print(f"HTTP hiba történt: {e}")
    print(f"Szerver válasza: {response.text}")
except Exception as e:
    print(f"Egyéb hiba történt: {e}")

In [ ]:
api_to_tippmix = {
    "Tampa Bay Rays": "Tampa Bay",
    "Baltimore Orioles": "Baltimore",
    "Pittsburgh Pirates": "Pittsburgh",
    "Texas Rangers": "Texas",
    "Detroit Tigers": "Detroit",
    "New York Yankees": "NY Yankees",
    "Kansas City Royals": "Kansas City",
    "Chicago White Sox": "Chicago WS",
    "Arizona Diamondbacks": "Arizona",
    "Houston Astros": "Houston",
    "St. Louis Cardinals": "St. Louis",
    "Colorado Rockies": "Colorado",
    "Los Angeles Dodgers": "LA Dodgers",
    "Los Angeles Angels": "LA Angels",
    "Cincinnati Reds": "Cincinnati",
    "Chicago Cubs": "Chicago Cubs",
    "Toronto Blue Jays": "Toronto",
    "Washington Nationals": "Washington",
    "New York Mets": "NY Mets",
    "Miami Marlins": "Miami",
    "Boston Red Sox": "Boston",
    "Philadelphia Phillies": "Philadelphia",
    "Cleveland Guardians": "Cleveland",
    "Minnesota Twins": "Minnesota",
    "Atlanta Braves": "Atlanta",
    "Milwaukee Brewers": "Milwaukee",
    "San Diego Padres": "San Diego",
    "San Francisco Giants": "San Francisco",
    "Seattle Mariners": "Seattle",
    "Athletics": "Las Vegas",
}

for i, game in games.iterrows():
    home = game['home_team_name']
    away = game['away_team_name']
    home_odds_name = api_to_tippmix[home]
    away_odds_name = api_to_tippmix[away]
    if home_odds_name not in df_odds['Home'].unique() or away_odds_name not in df_odds['Away'].unique():
        continue
    home_odds = df_odds.loc[(df_odds['Home'] == home_odds_name)]['H_odds'].iloc[0]
    away_odds = df_odds.loc[(df_odds['Away'] == away_odds_name)]['A_odds'].iloc[0]
    
    games.loc[i, 'home_fair_odds'] = home_odds
    games.loc[i, 'away_fair_odds'] = away_odds

    home_implied, away_implied = game['home_implied_prob'], game['away_implied_prob']
    home_value = home_odds > home_implied
    games.loc[i, 'home_value'] = home_value
    away_value = away_odds > away_implied
    games.loc[i, 'away_value'] = away_value

    print(f"\n{home} ({home_implied:.2f} - {home_value}) vs {away} ({away_implied:.2f} - {away_value})")

print(f"\n\n{len(games[pd.notna(games["away_fair_odds"])])}/{len(games)} games found")